# ETA Model Training (GPU)

Train the neural net on full 37M rows using T4 GPU.
Works on both Google Colab and Kaggle.

In [ ]:
# 1. Install dependencies (torch is pre-installed on Colab/Kaggle)
!pip install -q huggingface_hub pyarrow tqdm

In [ ]:
# 2. Clone repo
!git clone https://github.com/sarthakbiswas97/eta-engine.git
%cd eta-engine

In [ ]:
# 3. Download data from HF Hub (skip this cell if HF data not uploaded yet)
# import os
# import shutil
# from huggingface_hub import hf_hub_download
#
# REPO_ID = "sarthakbiswas/nyc-eta-data"
# os.makedirs("data/zone_pair_stats", exist_ok=True)
#
# files = {
#     "train.parquet": "data/train.parquet",
#     "dev.parquet": "data/dev.parquet",
#     "zone_pair_stats.pkl": "data/zone_pair_stats/zone_pair_stats.pkl",
# }
#
# for remote_name, local_path in files.items():
#     if os.path.exists(local_path):
#         print(f"Cached: {local_path}")
#         continue
#     print(f"Downloading {remote_name}...")
#     downloaded = hf_hub_download(
#         repo_id=REPO_ID,
#         filename=remote_name,
#         repo_type="dataset",
#     )
#     shutil.copy2(downloaded, local_path)
#     print(f"  -> {local_path}")

In [ ]:
# 3b. Download data directly from NYC TLC + compute zone-pair stats
# (Use this if HF data isn't uploaded yet)
!pip install -q -r requirements.txt
!python data/download_data.py
!python -m features.zone_pair_stats

In [ ]:
# 4. Verify GPU and data
import torch
import os

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU: {gpu}")
    print(f"VRAM: {vram:.1f} GB")
    cap = torch.cuda.get_device_capability(0)
    print(f"Compute capability: {cap[0]}.{cap[1]}")

print()
for f in ["data/train.parquet", "data/dev.parquet", "data/zone_pair_stats/zone_pair_stats.pkl"]:
    size = os.path.getsize(f) / 1e6 if os.path.exists(f) else 0
    status = f"{size:.1f} MB" if size > 0 else "MISSING"
    print(f"  {f}: {status}")

In [ ]:
# 5. Train v2: new features (24) + Huber loss
!python train.py --epochs 10 --batch-size 8192 --lr 5e-4 --patience 3 --num-workers 2 --dev-sample 50000 --save-every 2 --loss huber

In [ ]:
# 6. Check results
checkpoint = torch.load("model.pt", map_location="cpu", weights_only=False)
print(f"Best dev MAE: {checkpoint['dev_mae']:.1f} s")
print(f"Best epoch: {checkpoint['epoch']}")
print(f"Model config: {checkpoint['model_config']}")

In [ ]:
# 7. Upload trained model to HF Hub
from huggingface_hub import HfApi

# Get HF token
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass

if token:
    api = HfApi(token=token)
    api.upload_file(
        path_or_fileobj="model.pt",
        path_in_repo="model.pt",
        repo_id="sarthakbiswas/nyc-eta-data",
        repo_type="dataset",
    )
    print("Model uploaded to HF Hub")
else:
    print("No HF_TOKEN found. Download model.pt manually from Files panel.")